# summarize

> Generate section and full summaries from video xscripts via LLM

In [ ]:
#| default_exp summarize

## Design

One LLM call generates all section summaries + a full video summary.
Each section summary includes: summary text, keywords, and evidence quote.

**summaries.json format:**
```json
{
  "full": {"summary": "...", "keywords": [...], "evidence": {"text": "...", "at": 123}},
  "sections": {
    "1": {"summary": "...", "keywords": [...], "evidence": {"text": "...", "at": 456}}
  }
}
```

If `toc.json` is missing, `generate_toc` is called first internally.

In [ ]:
#| export
import json
from pathlib import Path

In [ ]:
#| export
def _slice_segments(segments: list[dict], # [{start, end, text}, ...]
                    start: int, # Section start in seconds
                    end: int # Section end in seconds
                   ) -> list[dict]: # Segments within [start, end)
    "Return segments that fall within the given time range."
    return [s for s in segments if s['start'] >= start and s['start'] < end]

def _build_summary_prompt(segments: list[dict], # Full xscript segments
                          sections: list[dict], # [{path, title, start, end}, ...] from toc.json
                          meta: dict # meta.json content
                         ) -> str: # Prompt for LLM
    "Build prompt asking LLM to summarize each section and the full video."
    parts = []
    for sec in sections:
        sliced = _slice_segments(segments, sec['start'], sec['end'])
        lines = []
        for s in sliced:
            mm = int(s['start'] // 60)
            ss = int(s['start'] % 60)
            lines.append(f"[{mm:02d}:{ss:02d}] {s['text']}")
        parts.append(f"### Section {sec['path']}: {sec['title']}\n" + '\n'.join(lines))

    transcript = '\n\n'.join(parts)
    title = meta.get('title', '')
    channel = meta.get('channel', '')
    desc = meta.get('description', '')

    return f"""You are a structural editor for YouTube video transcripts.

Video info:
- Title: {title}
- Channel: {channel}
- Description: {desc}

Transcript (organized by section):
{transcript}

Task:
For each section AND for the full video, provide:
- summary: 1-2 sentence English summary
- keywords: important terms (people, technical terms, proper nouns)
- evidence: a short quoted phrase from the transcript with its timestamp in seconds

Return a JSON object with:
- "full": {{summary, keywords, evidence: {{text, at}}}}
- "sections": {{"1": {{summary, keywords, evidence: {{text, at}}}}, "2": ...}}"""

In [ ]:
#| export
_SUMMARY_SCHEMA = {
    "type": "object",
    "properties": {
        "full": {
            "type": "object",
            "properties": {
                "summary": {"type": "string"},
                "keywords": {"type": "array", "items": {"type": "string"}},
                "evidence": {
                    "type": "object",
                    "properties": {"text": {"type": "string"}, "at": {"type": "integer"}},
                    "required": ["text", "at"],
                },
            },
            "required": ["summary", "keywords", "evidence"],
        },
        "sections": {
            "type": "object",
            "additionalProperties": {
                "type": "object",
                "properties": {
                    "summary": {"type": "string"},
                    "keywords": {"type": "array", "items": {"type": "string"}},
                    "evidence": {
                        "type": "object",
                        "properties": {"text": {"type": "string"}, "at": {"type": "integer"}},
                        "required": ["text", "at"],
                    },
                },
                "required": ["summary", "keywords", "evidence"],
            },
        },
    },
    "required": ["full", "sections"],
}

def _call_summary_llm(prompt: str) -> dict:
    "Call OpenAI gpt-5.4 with structured output, return {full, sections} dict."
    import openai
    client = openai.OpenAI()
    response = client.chat.completions.create(
        model='gpt-5.4',
        response_format={
            "type": "json_schema",
            "json_schema": {"name": "generate_summaries", "schema": _SUMMARY_SCHEMA},
        },
        messages=[{"role": "user", "content": prompt}],
    )
    return json.loads(response.choices[0].message.content)

## Tests

In [ ]:
# Test 1: _slice_segments returns segments within time range
segs = [
    {'start': 0, 'end': 5, 'text': 'a'},
    {'start': 5, 'end': 10, 'text': 'b'},
    {'start': 10, 'end': 15, 'text': 'c'},
    {'start': 15, 'end': 20, 'text': 'd'},
]
sliced = _slice_segments(segs, start=5, end=15)
assert len(sliced) == 2
assert sliced[0]['text'] == 'b'
assert sliced[1]['text'] == 'c'
print('ok')

In [ ]:
# Test 2: _slice_segments with no matching segments returns empty
sliced = _slice_segments(segs, start=100, end=200)
assert sliced == []
print('ok')

In [ ]:
# Test 3: _build_summary_prompt includes section titles and transcript
segments = [
    {'start': 0, 'end': 5, 'text': 'hello world'},
    {'start': 5, 'end': 10, 'text': 'second part'},
]
sections = [
    {'path': '1', 'title': 'Intro', 'start': 0, 'end': 5},
    {'path': '2', 'title': 'Main', 'start': 5, 'end': 10},
]
meta = {'title': 'Test Video', 'channel': 'Ch'}
prompt = _build_summary_prompt(segments, sections, meta)
assert 'Test Video' in prompt
assert 'Intro' in prompt
assert 'Main' in prompt
assert 'hello world' in prompt
assert 'second part' in prompt
print('ok')

## CLI

In [ ]:
#| export
from fastcore.script import call_parse
from yttoc.core import fmt_duration, format_header
from yttoc.fetch import _DEFAULT_ROOT, _update_last_used, _glob_srt
from yttoc.xscript import parse_xscript
from yttoc.toc import generate_toc

def generate_summaries(video_id: str, # Exact video_id
                       root: Path = None, # Root cache directory
                      ) -> dict: # {full, sections} summaries
    "Generate summaries.json for a cached video. Returns summaries dict."
    root = root or _DEFAULT_ROOT
    d = root / video_id
    meta_path = d / 'meta.json'
    sum_path = d / 'summaries.json'
    srt_files = _glob_srt(d)
    if not (meta_path.exists() and srt_files):
        raise SystemExit(f"Not cached: {video_id}")

    # Return cached summaries if exists
    if sum_path.exists():
        return json.loads(sum_path.read_text(encoding='utf-8'))

    # Ensure toc.json exists
    sections = generate_toc(video_id, root)

    meta = json.loads(meta_path.read_text(encoding='utf-8'))
    segments = parse_xscript(srt_files[0])
    prompt = _build_summary_prompt(segments, sections, meta)
    result = _call_summary_llm(prompt)

    sum_path.write_text(
        json.dumps(result, indent=2, ensure_ascii=False),
        encoding='utf-8')
    _update_last_used(meta_path)
    return result

@call_parse
def yttoc_sum(video_id: str, # Exact video_id
              section: str = None, # Optional section path (e.g. "3")
              root: str = None, # Root cache directory
             ):
    "Display summaries for a cached video."
    root = Path(root) if root else _DEFAULT_ROOT
    d = root / video_id
    meta_path = d / 'meta.json'
    if not meta_path.exists():
        raise SystemExit(f"Not cached: {video_id}")

    meta = json.loads(meta_path.read_text(encoding='utf-8'))
    sums = generate_summaries(video_id, root)

    print(format_header(meta))
    print()

    if section:
        # Show specific section
        s = sums['sections'].get(section)
        if s is None:
            raise SystemExit(f"Section {section} not found")
        toc = json.loads((d / 'toc.json').read_text(encoding='utf-8'))
        sec_info = next((t for t in toc['sections'] if t['path'] == section), None)
        title = sec_info['title'] if sec_info else f'Section {section}'
        start = fmt_duration(sec_info['start']) if sec_info else '?'
        print(f"## {section}. {title} ({start})")
        print(s['summary'])
        print(f"**Keywords:** {', '.join(s['keywords'])}")
        print(f"**Evidence:** \"{s['evidence']['text']}\" [{fmt_duration(s['evidence']['at'])}]")
    else:
        # Show all sections + full
        for path, s in sorted(sums['sections'].items(), key=lambda x: int(x[0])):
            toc = json.loads((d / 'toc.json').read_text(encoding='utf-8'))
            sec_info = next((t for t in toc['sections'] if t['path'] == path), None)
            title = sec_info['title'] if sec_info else f'Section {path}'
            start = fmt_duration(sec_info['start']) if sec_info else '?'
            print(f"## {path}. {title} ({start})")
            print(s['summary'])
            print(f"**Keywords:** {', '.join(s['keywords'])}")
            print(f"**Evidence:** \"{s['evidence']['text']}\" [{fmt_duration(s['evidence']['at'])}]")
            print()

        print("## Full Summary")
        print(sums['full']['summary'])
        print(f"**Keywords:** {', '.join(sums['full']['keywords'])}")
        print(f"**Evidence:** \"{sums['full']['evidence']['text']}\" [{fmt_duration(sums['full']['evidence']['at'])}]")

In [ ]:
# Test 4: generate_summaries returns cached summaries.json without LLM call
from tempfile import TemporaryDirectory
import io, contextlib

_test_summaries = {
    'full': {'summary': 'Full video about testing.', 'keywords': ['test'], 'evidence': {'text': 'hello', 'at': 0}},
    'sections': {
        '1': {'summary': 'Intro section.', 'keywords': ['intro'], 'evidence': {'text': 'hi', 'at': 0}},
        '2': {'summary': 'Main section.', 'keywords': ['main'], 'evidence': {'text': 'bye', 'at': 300}},
    },
}

with TemporaryDirectory() as d:
    root = Path(d)
    v = root / 'VID1'; v.mkdir()
    (v / 'captions.en.srt').write_text('1\n00:00:00,000 --> 00:00:01,000\nhi\n')
    (v / 'meta.json').write_text(json.dumps({
        'id': 'VID1', 'title': 'T', 'channel': 'C', 'duration': 600,
        'upload_date': '20260101', 'last_used_at': '2000-01-01T00:00:00'}))
    (v / 'toc.json').write_text(json.dumps({'sections': [
        {'path': '1', 'title': 'Intro', 'start': 0, 'end': 300},
        {'path': '2', 'title': 'Main', 'start': 300, 'end': 600},
    ]}))
    (v / 'summaries.json').write_text(json.dumps(_test_summaries))

    result = generate_summaries('VID1', root)
    assert result['full']['summary'] == 'Full video about testing.'
    assert '1' in result['sections']
    assert '2' in result['sections']
print('ok')

In [ ]:
# Test 5: yttoc_sum displays all sections + full summary
with TemporaryDirectory() as d:
    root = Path(d)
    v = root / 'VID2'; v.mkdir()
    (v / 'captions.en.srt').write_text('1\n00:00:00,000 --> 00:00:01,000\nhi\n')
    (v / 'meta.json').write_text(json.dumps({
        'id': 'VID2', 'title': 'Test Video', 'channel': 'Ch', 'duration': 600,
        'upload_date': '20260101', 'webpage_url': 'https://youtube.com/watch?v=VID2',
        'last_used_at': '2000-01-01T00:00:00'}))
    (v / 'toc.json').write_text(json.dumps({'sections': [
        {'path': '1', 'title': 'Intro', 'start': 0, 'end': 300},
        {'path': '2', 'title': 'Main', 'start': 300, 'end': 600},
    ]}))
    (v / 'summaries.json').write_text(json.dumps(_test_summaries))

    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        yttoc_sum('VID2', root=str(root))
    out = buf.getvalue()

    assert '# Test Video' in out
    assert '## 1. Intro (0:00)' in out
    assert 'Intro section.' in out
    assert '**Keywords:** intro' in out
    assert '## Full Summary' in out
    assert 'Full video about testing.' in out
print('ok')

In [ ]:
# Test 6: yttoc_sum with section argument shows only that section
with TemporaryDirectory() as d:
    root = Path(d)
    v = root / 'VID3'; v.mkdir()
    (v / 'captions.en.srt').write_text('1\n00:00:00,000 --> 00:00:01,000\nhi\n')
    (v / 'meta.json').write_text(json.dumps({
        'id': 'VID3', 'title': 'T', 'channel': 'C', 'duration': 600,
        'upload_date': '20260101', 'last_used_at': '2000-01-01T00:00:00'}))
    (v / 'toc.json').write_text(json.dumps({'sections': [
        {'path': '1', 'title': 'Intro', 'start': 0, 'end': 300},
        {'path': '2', 'title': 'Main', 'start': 300, 'end': 600},
    ]}))
    (v / 'summaries.json').write_text(json.dumps(_test_summaries))

    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        yttoc_sum('VID3', section='2', root=str(root))
    out = buf.getvalue()

    assert '## 2. Main (5:00)' in out
    assert 'Main section.' in out
    assert 'Intro section.' not in out  # only section 2
    assert '## Full Summary' not in out
print('ok')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()